In [1]:
import duckdb

DATA_DIR = r"C:\Users\shyam\Downloads\synerise_dataset"

con = duckdb.connect("ecommerce_analytics.duckdb")

print("=" * 80)
print("BUILDING CUSTOMER BEHAVIOR TABLE")
print("=" * 80)

query = f"""
CREATE OR REPLACE TABLE customer_behavior AS

WITH

page AS (
    SELECT
        client_id,
        COUNT(*) AS page_visits,
        COUNT(DISTINCT CAST(timestamp AS DATE)) AS page_active_days,
        MIN(CAST(timestamp AS TIMESTAMP)) AS first_page_visit,
        MAX(CAST(timestamp AS TIMESTAMP)) AS last_page_visit
    FROM read_parquet('{DATA_DIR}/page_visit.parquet')
    GROUP BY client_id
),

searches AS (
    SELECT
        client_id,
        COUNT(*) AS searches,
        COUNT(DISTINCT CAST(timestamp AS DATE)) AS search_active_days,
        MIN(CAST(timestamp AS TIMESTAMP)) AS first_search,
        MAX(CAST(timestamp AS TIMESTAMP)) AS last_search
    FROM read_parquet('{DATA_DIR}/search_query.parquet')
    GROUP BY client_id
),

cart_add AS (
    SELECT
        client_id,
        COUNT(*) AS cart_additions,
        COUNT(DISTINCT sku) AS unique_skus_added,
        COUNT(DISTINCT CAST(timestamp AS DATE)) AS cart_add_active_days,
        MIN(CAST(timestamp AS TIMESTAMP)) AS first_cart_add,
        MAX(CAST(timestamp AS TIMESTAMP)) AS last_cart_add
    FROM read_parquet('{DATA_DIR}/add_to_cart.parquet')
    GROUP BY client_id
),

cart_remove AS (
    SELECT
        client_id,
        COUNT(*) AS cart_removals,
        COUNT(DISTINCT sku) AS unique_skus_removed,
        COUNT(DISTINCT CAST(timestamp AS DATE)) AS cart_remove_active_days,
        MIN(CAST(timestamp AS TIMESTAMP)) AS first_cart_remove,
        MAX(CAST(timestamp AS TIMESTAMP)) AS last_cart_remove
    FROM read_parquet('{DATA_DIR}/remove_from_cart.parquet')
    GROUP BY client_id
),

purchase AS (
    SELECT
        client_id,
        COUNT(*) AS purchases,
        COUNT(DISTINCT sku) AS unique_skus_purchased,
        COUNT(DISTINCT CAST(timestamp AS DATE)) AS purchase_active_days,
        MIN(CAST(timestamp AS TIMESTAMP)) AS first_purchase,
        MAX(CAST(timestamp AS TIMESTAMP)) AS last_purchase
    FROM read_parquet('{DATA_DIR}/product_buy.parquet')
    GROUP BY client_id
),

all_activity AS (

    SELECT
        client_id,
        CAST(timestamp AS TIMESTAMP) AS event_time
    FROM read_parquet('{DATA_DIR}/page_visit.parquet')

    UNION ALL

    SELECT
        client_id,
        CAST(timestamp AS TIMESTAMP)
    FROM read_parquet('{DATA_DIR}/search_query.parquet')

    UNION ALL

    SELECT
        client_id,
        CAST(timestamp AS TIMESTAMP)
    FROM read_parquet('{DATA_DIR}/add_to_cart.parquet')

    UNION ALL

    SELECT
        client_id,
        CAST(timestamp AS TIMESTAMP)
    FROM read_parquet('{DATA_DIR}/remove_from_cart.parquet')

    UNION ALL

    SELECT
        client_id,
        CAST(timestamp AS TIMESTAMP)
    FROM read_parquet('{DATA_DIR}/product_buy.parquet')
),

activity AS (
    SELECT
        client_id,
        COUNT(DISTINCT CAST(event_time AS DATE)) AS active_days,
        MIN(event_time) AS first_activity,
        MAX(event_time) AS last_activity
    FROM all_activity
    GROUP BY client_id
)

SELECT

    a.client_id,

    -- Overall activity
    a.active_days,
    a.first_activity,
    a.last_activity,

    -- Page behavior
    COALESCE(p.page_visits, 0) AS page_visits,

    -- Search behavior
    COALESCE(s.searches, 0) AS searches,

    -- Cart behavior
    COALESCE(ca.cart_additions, 0) AS cart_additions,
    COALESCE(cr.cart_removals, 0) AS cart_removals,

    COALESCE(ca.unique_skus_added, 0) AS unique_skus_added,
    COALESCE(cr.unique_skus_removed, 0) AS unique_skus_removed,

    -- Purchase behavior
    COALESCE(pr.purchases, 0) AS purchases,
    COALESCE(pr.unique_skus_purchased, 0) AS unique_skus_purchased,

    -- Dates
    pr.first_purchase,
    pr.last_purchase,

    -- Derived metrics
    CASE
        WHEN COALESCE(ca.cart_additions, 0) > 0
        THEN ROUND(
            COALESCE(cr.cart_removals, 0) * 1.0
            / ca.cart_additions,
            4
        )
        ELSE 0
    END AS cart_removal_rate,

    CASE
        WHEN COALESCE(ca.cart_additions, 0) > 0
        THEN ROUND(
            COALESCE(pr.purchases, 0) * 1.0
            / ca.cart_additions,
            4
        )
        ELSE 0
    END AS purchase_per_cart_rate,

    CASE
        WHEN COALESCE(s.searches, 0) > 0
        THEN ROUND(
            COALESCE(pr.purchases, 0) * 1.0
            / s.searches,
            4
        )
        ELSE 0
    END AS purchase_per_search_rate,

    CASE
        WHEN COALESCE(p.page_visits, 0) > 0
        THEN ROUND(
            COALESCE(pr.purchases, 0) * 1.0
            / p.page_visits,
            6
        )
        ELSE 0
    END AS purchase_per_page_rate

FROM activity a

LEFT JOIN page p
    ON a.client_id = p.client_id

LEFT JOIN searches s
    ON a.client_id = s.client_id

LEFT JOIN cart_add ca
    ON a.client_id = ca.client_id

LEFT JOIN cart_remove cr
    ON a.client_id = cr.client_id

LEFT JOIN purchase pr
    ON a.client_id = pr.client_id
"""

con.execute(query)

print("\nCustomer behavior table created successfully!")

# ============================================================
# TABLE SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("TABLE SUMMARY")
print("=" * 80)

result = con.execute("""
SELECT
    COUNT(*) AS customers,
    AVG(active_days) AS avg_active_days,
    AVG(page_visits) AS avg_page_visits,
    AVG(searches) AS avg_searches,
    AVG(cart_additions) AS avg_cart_additions,
    AVG(cart_removals) AS avg_cart_removals,
    AVG(purchases) AS avg_purchases
FROM customer_behavior
""").fetchdf()

print(result.to_string(index=False))


# ============================================================
# TOP CUSTOMERS
# ============================================================

print("\n" + "=" * 80)
print("TOP CUSTOMERS BY PURCHASES")
print("=" * 80)

result = con.execute("""
SELECT
    client_id,
    active_days,
    page_visits,
    searches,
    cart_additions,
    cart_removals,
    purchases,
    unique_skus_purchased
FROM customer_behavior
ORDER BY purchases DESC
LIMIT 20
""").fetchdf()

print(result.to_string(index=False))


# ============================================================
# PURCHASE VS NON-PURCHASE CUSTOMERS
# ============================================================

print("\n" + "=" * 80)
print("PURCHASERS VS NON-PURCHASERS")
print("=" * 80)

result = con.execute("""
SELECT
    CASE
        WHEN purchases > 0 THEN 'Purchaser'
        ELSE 'Non-Purchaser'
    END AS customer_type,

    COUNT(*) AS customers,

    ROUND(AVG(active_days), 2) AS avg_active_days,
    ROUND(AVG(page_visits), 2) AS avg_page_visits,
    ROUND(AVG(searches), 2) AS avg_searches,
    ROUND(AVG(cart_additions), 2) AS avg_cart_additions,
    ROUND(AVG(cart_removals), 2) AS avg_cart_removals,
    ROUND(AVG(purchases), 2) AS avg_purchases

FROM customer_behavior

GROUP BY customer_type
ORDER BY customer_type
""").fetchdf()

print(result.to_string(index=False))

con.close()

print("\nDONE!")

BUILDING CUSTOMER BEHAVIOR TABLE

Customer behavior table created successfully!

TABLE SUMMARY
 customers  avg_active_days  avg_page_visits  avg_searches  avg_cart_additions  avg_cart_removals  avg_purchases
  22298361         2.103122         8.944692      0.593038            0.338192           0.120587       0.103976

TOP CUSTOMERS BY PURCHASES
 client_id  active_days  page_visits  searches  cart_additions  cart_removals  purchases  unique_skus_purchased
  10541597            5         1398       236             674             26        644                     20
   5358489          126         3713       245             617             20        536                      6
  17761854          122         4232       920             438             16        485                     35
  23047216           52         1070       181             158             24        404                    282
   7410183           57         6029       604             776            326        383   